# demo_clue

> Phase 11 (`specs/multiagent/phase11/`): the clue board, where every subtask's answer is fixed by modular clues dealt to different agents, so sharing is needed to solve it. Numpy sweep, then LLM pilot and main runs through the faster scheduler.

Artifacts under `results/multiagent/clue/{quick,sweep,pilot,full}/` (gitignored): `runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md` and figures. The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), so one cap ($150) covers all live spend.

- **Quick** (normal tests): every clue path with the fake client; outputs identical for 1 and 8 processes, pooled and grid by grid.
- **Sweep** (`--cell-timeout 0 --flags slow`): rule agents over the clue settings; the frozen rule picks the settings.
- **Pilot** and **main** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live, the main runs only after `CLUE_PLAN` is frozen.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import readCSV, runGrid, writeCSV
from explRL.clue_grid import (CLUE_GRIDS, CLUE_OPTIONAL, CLUE_PLAN, CLUE_SETTINGS, chooseClueSettings, clueHash,
                              clueNumpyGrid, cluePilotGrid, clueQuick, clueSweep, sweepIncentives)
from explRL.clue_report import summaries, writeFigures, writeReport
from explRL.clue_runs import runClueAny
from explRL.llm_budget import PRICES, Budget
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.train import demo_artifact_dir

DN = 'multiagent/clue'
WORKERS = min(8, os.cpu_count() or 1)          # fake-client and numpy workloads
LLM_WORKERS = 32                               # live workloads (as the follow-up runs)
OPTIONAL = False                               # effort and scale arms (K6: off)
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
SETTINGS = ('m', 'r', 'R', 'guess')


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def run(mode, grids, spec, workers=None):
    "Run a workload through runWorkload; write runs.csv, turns.jsonl, timing.jsonl, report.md, figures, metrics.json."
    out = demo_artifact_dir(mode, DN)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    workers = workers or (LLM_WORKERS if spec.get('kind') == 'live' else WORKERS)
    t0 = time.time()
    res = runWorkload(grids, spec, workers, str(logs), run=runClueAny)
    rows, wall = res['rows'], time.time() - t0
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    llm = [r for r in rows if r['runner'] == 'llm']
    models = sorted({r['model'] for r in llm})
    meta = dict(mode=mode, planHash=clueHash(), commit=gitCommit(), models=models, cap=CAP, wall=wall,
                settings=', '.join(f'{k} = {CLUE_SETTINGS[k]}' for k in SETTINGS),
                pricesChecked=', '.join(sorted({PRICES[m]['checked'] for m in models})) or '-',
                cost=float(sum(r['cost'] for r in llm)), calls=int(sum(r['calls'] for r in llm)), retries=timing['retries'],
                spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    S = summaries(llm, [r for r in rows if r['study'] == 'clueNumpy'])
    writeReport(out / 'report.md', S, meta)
    writeFigures(out, S)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      pooled=res['pooled'], workers=workers, timing=timing,
                                                      plan=CLUE_PLAN), indent=2, sort_keys=True, default=str) + '\n')
    if spec.get('kind') == 'live':             # the first (live) run's metrics and timing; a cached rerun overwrites the others
        for f in ('metrics.json', 'timing.jsonl'):
            live = out / f.replace('.', '_live.')
            if not live.exists():
                shutil.copy(out / f, live)
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent "
          f"in all, {wall:.0f} s, {timing['callsPerSecond']:.1f} calls/s, {timing['retries']} retries; stopped: {res['stopped']}")
    return rows, S, meta, timing

## Quick: every clue path with the fake client (pipeline check)

In [ ]:
t0 = time.time()
rows, S, meta, timing = run('quick', [('quick', clueQuick())], dict(kind='fake'))
assert time.time() - t0 < 30
q = demo_artifact_dir('quick', DN)
for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', 'clue_welfare.png', 'clue_info.png',
          'clue_numpy.png', 'clue_arms.png'):
    assert (q / f).stat().st_size > 0, f
ref = (q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()
for w, pooled in ((1, True), (8, False)):
    run('quick', [('quick', clueQuick())], dict(kind='fake', pooled=pooled), workers=w)
    assert ((q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()) == ref
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:1200])

## Sweep (`--cell-timeout 0 --flags slow`): rule agents pick the settings

Rule agents over the (m, R) pairs x clues per holder r x guess rate x sharing ('needed' or 'never') x N 1-64 x channel, seeds 800-819 (22 400 runs, a few minutes on 8 processes). The frozen rule (FR-B3) picks the settings at N = 4: blackboard welfare at most 0.8 of the pooled bound and no-channel welfare at most half the blackboard's, the largest gap wins. If none qualifies the phase stops.

In [ ]:
#| slow
t0 = time.time()
srows = runGrid(clueSweep(), WORKERS, run=runClueAny)
writeCSV(srows, demo_artifact_dir('sweep', DN) / 'runs.csv')
srows = readCSV(demo_artifact_dir('sweep', DN) / 'runs.csv')
chosen = chooseClueSettings(srows)
print(f'{len(srows)} runs in {time.time() - t0:.0f} s; settings (frozen rule):', chosen)
assert chosen is not None, 'no setting qualifies: stop and report (T5)'
print('incentives:', json.dumps(sweepIncentives(srows, chosen), indent=1, default=str))

## Pilot (`--flags llm`, live, about 540 calls): solving, interface and token profile

Blackboard and no channel at N = 4, and N = 1 (one agent holds every clue), seeds 1100-1101, 30 turns. AC-9: at N = 1 Haiku solves at least 80% of the subtasks it opens, with under 5% invalid actions. The measured token profile replaces the provisional `('claude-haiku-5-5', 'clue')` entry before `CLUE_PLAN` is frozen.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    chosen = chooseClueSettings(readCSV(demo_artifact_dir('sweep', DN) / 'runs.csv'))
    assert chosen and all(CLUE_SETTINGS[k] == chosen[k] for k in SETTINGS), 'write the sweep settings into CLUE_SETTINGS first'
    prows, _, _, ptiming = run('pilot', [('pilot', cluePilotGrid())], LIVE_SPEC)
    one = [r for r in prows if int(r['N']) == 1]
    calls = sum(r['calls'] for r in prows)
    solve = sum(r['solved'] for r in one) / max(sum(r['opens'] for r in one), 1)
    invalid = sum(r['invalid'] for r in prows) / max(calls, 1)
    profile = tuple(round(sum(r[k] for r in prows) / max(calls, 1)) for k in ('inputTokens', 'cacheReadTokens', 'outputTokens'))
    print(f'solve rate at N = 1: {solve:.2f} (AC-9: >= 0.8); invalid {invalid:.3f} (< 0.05); token profile {profile}')

## Main (`--cell-timeout 0 --flags llm`, live, about $6-8): main, framing and liar in one workload

Priority order main, framing, liar (used only if the workload does not fit the cap), with the 60 rule-agent cells for the numpy comparison; the effort and scale arms only with `OPTIONAL = True`.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert CLUE_PLAN['frozen'], 'freeze and commit CLUE_PLAN before the main runs'
    grids = [(n, g()) for n, g in CLUE_GRIDS.items()] + [('numpy', clueNumpyGrid())]
    grids += [(n, g()) for n, g in CLUE_OPTIONAL.items()] if OPTIONAL else []
    rows, S, meta, timing = run('full', grids, LIVE_SPEC)
    print(open(demo_artifact_dir('full', DN) / 'report.md').read()[:3000])